In [1]:
!pip install langchain pypdf --quiet
!pip install langchain-community --quiet

In [2]:
pip install pypdf2


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
from transformers import pipeline
import torch
from langchain.document_loaders import PyPDFLoader
from tqdm import tqdm
import re

RuntimeError: Failed to import transformers.pipelines because of the following error (look up to see its traceback):
No module named 'torch._custom_ops'

In [ ]:
import os
from PyPDF2 import PdfMerger

# List of PDF files to merge (provide the paths to your 4 PDFs)
pdf_files = [
    "SecuritiesandCommoditiesExchangeMarketRelatedLaws.pdf",
    "SpecializedInvestmentFundRules.pdf",
    "StrategicPlan.pdf",
    "booklet.pdf",
]

# Initialize the PdfMerger object
merger = PdfMerger()

# Loop through the PDF files and append them if they exist
for pdf in pdf_files:
    if os.path.exists(pdf):
        merger.append(pdf)
    else:
        print(f"File not found: {pdf}")

# Specify the output file name
output_file = "merged_document.pdf"

# Write the merged PDF to the output file
merger.write(output_file)
merger.close()

print(f"All PDFs have been merged into {output_file}")


All PDFs have been merged into merged_document.pdf


In [ ]:

# Load the merged PDF
pdf_name = "merged_document.pdf"
loader = PyPDFLoader(pdf_name)

# Split the PDF into pages
pages = loader.load_and_split()

print(f"Loaded {len(pages)} pages from the merged PDF.")

Loaded 998 pages from the merged PDF.


In [ ]:
def clean_dataset(entire_text):
    """
    Cleans the text content of a document page by performing multiple preprocessing steps:
    - Removes extra whitespaces.
    - Fixes hyphenated words.
    - Removes headers, footers, page numbers, and non-informative text.
    - Extracts relevant sections.
    - Normalizes and standardizes text.
    - Cleans unnecessary legal jargon.
    """
    text = entire_text.page_content

    # Step 1: Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Step 2: Fix hyphenated words (joins split words)
    text = re.compile(r'(\w+)\s*-\s*(\w+)').sub(lambda match: match.group(1) + match.group(2), text)

    # Step 3: Remove non-informative headers/footers like page numbers and ToC references
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)  # Page numbers
    text = re.sub(r"Table of Content.*", "", text, flags=re.IGNORECASE)  # ToC references

    # Step 4: Normalize and standardize text
    text = text.lower()  # Convert to lowercase for consistency
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')  # Replace unusual quotation marks

    # Step 5: Remove redundant legal formatting
    text = re.sub(r"provided that.*?\.", "", text, flags=re.IGNORECASE)  # Remove "Provided that..." phrases

    # Step 6: Extract relevant sections (optional, if specific keywords are required)
    relevant_sections = ["securities act", "market regulation", "money laundering"]
    is_relevant = any(section in text for section in relevant_sections)

    return entire_text if is_relevant else None

In [ ]:
# Clean each page
cleaned_pages = [page for page in (clean_dataset(page) for page in pages) if page is not None]

# Check the first cleaned page for verification
print(cleaned_pages[0].page_content)


7
1. Acts
A. Securities and Commodities Market 
i. Securities Act, 2006 (2063)
ii. Commodities Exchange Market Act, 2017 (2074)
B. Money Laundering
i. Asset (Money) Laundering Prevention Act, 2008 (2064)


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Join all cleaned pages into one string
cleaned_text = "\n".join(page.page_content for page in cleaned_pages)

# Set up the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = text_splitter.split_text(cleaned_text)


# Check the first chunk for verification
print(chunks[0])


7
1. Acts
A. Securities and Commodities Market 
i. Securities Act, 2006 (2063)
ii. Commodities Exchange Market Act, 2017 (2074)
B. Money Laundering
i. Asset (Money) Laundering Prevention Act, 2008 (2064)
1
SECURITIES ACT, 2006
          Date of Authentication and 
Publication: 14/01/ 2007 
                Some Nepal Act Amendment Act, 2016: 25/02/2016
Act Number 33 of the year 2007/2008
An Act Made to Amend and Consolidate Laws Relating to Securities 
Preamble: Whereas, it is expedient to make timely the laws relating to securities by 
amending and consolidating such laws in order to regulate and manage the activities 
of the securities markets and  persons involved in the business of dealing in securities 
by regulating the issuance, purchase, sale and exchange of securities for the purpose 
of protecting the interests of investors in securities, by developing the capital market 
to mobilize necessary capital for the economic development of the country; Now,


In [ ]:
len(chunks)

249

In [ ]:
!pip install sentence-transformers --quiet
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
sentence_transformer = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-mpnet-base-v2',
    model_kwargs={'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu')}
)


In [ ]:
!pip install faiss-cpu


In [ ]:
from langchain.vectorstores import FAISS

In [ ]:
from langchain.schema import Document
# Convert chunks (strings) into Document objects
documents = [
    Document(
        page_content=chunk, 
        metadata={"source": "merged_document.pdf", "page": idx}
    ) 
    for idx, chunk in enumerate(chunks)
]


In [ ]:
%%time

# Creating and storing embeddings in the FAISS vector store
vector_db = FAISS.from_documents(documents, sentence_transformer)

CPU times: user 4.94 s, sys: 151 ms, total: 5.09 s
Wall time: 5.59 s


In [ ]:
vector_db.save_local("final_ready_vector_db_data")

In [ ]:

# Load the FAISS vector store
docsearch = FAISS.load_local(
    "final_ready_vector_db_data",
    sentence_transformer,
    allow_dangerous_deserialization=True
)

# Perform a similarity search
query = "What is NEPSE?"
results = docsearch.similarity_search(query, k=5)

# Show results with metadata
for result in results:
    print(f"Page {result.metadata['page']} from {result.metadata['source']}:")
    print(result.page_content)

c) rules mean the rules framed under the act. d) nepse means the nepal stock exchange ltd. licensed by the board to run securities market. e) body corporate means a body corporate enliste d for securities at the nepse. f) institutional activities mean activities such as closing the registry book of a body corporate, organize general meetings, make retur n payments for the time expired s ecurities, revise the debentures and title deeds having convertible features, disbursing dividends, interest, bonus shares, right shares and priority shares, issue and pay f or the title deeds, return payment of premium and other associated tasks. g) institutional benefits mean benefits such as disbursing dividends, interest, bonus shares, rights shares, priority shares, issuing title deeds and making return payment of premium and other associated perks. h) enlistment means an enlistment made at the nepse for the purpose of purchase, sale or exchange of securities through the securities market
c) rules 